# World Cup 2026 Forecast Model
Based on Groll et al. (2019) weighted Poisson ranking model.

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.linear_model import PoissonRegressor
from scipy.optimize import linear_sum_assignment
from scipy.stats import poisson

print('Dependencies loaded.')

In [ ]:
# Download historical international match results
df = pd.read_csv('https://raw.githubusercontent.com/martj42/international_results/master/results.csv')
df.to_csv('results.csv', index=False)
print(f'Downloaded {len(df):,} matches ({df["date"].min()} to {df["date"].max()})')

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────────
RNG = np.random.default_rng(20260614)

HALF_PERIOD_DAYS = 3 * 365
WINDOW_YEARS     = 8
HOSTS            = {'United States', 'Canada', 'Mexico'}

def importance_weight(tournament: str) -> float:
    t = tournament.lower()
    if t == 'fifa world cup':                          return 4.0
    if 'qualification' in t or 'qualifier' in t:       return 2.5
    finals = ['uefa euro', 'copa am', 'african cup of nations', 'afc asian cup',
              'gold cup', 'confederations', 'nations league', 'concacaf championship',
              'oceania nations']
    if any(f in t for f in finals) and 'qualification' not in t: return 3.0
    if t == 'friendly':                                return 1.0
    return 1.5

def load_matches(path='results.csv'):
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.dropna(subset=['home_score', 'away_score']).copy()
    df['home_score'] = df['home_score'].astype(int)
    df['away_score'] = df['away_score'].astype(int)
    return df

def fit_strengths(df, ref_date, ridge=1e-4):
    cutoff = ref_date - pd.Timedelta(days=WINDOW_YEARS * 365)
    d = df[(df['date'] > cutoff) & (df['date'] <= ref_date)].copy()
    att   = pd.concat([d['home_team'], d['away_team']], ignore_index=True)
    deff  = pd.concat([d['away_team'], d['home_team']], ignore_index=True)
    goals = pd.concat([d['home_score'], d['away_score']], ignore_index=True).astype(float).values
    home  = np.concatenate([np.where(d['neutral'], 0, 1), np.zeros(len(d))]).astype(float)
    days  = np.concatenate([(ref_date - d['date']).dt.days.values] * 2)
    w_time = 0.5 ** (days / HALF_PERIOD_DAYS)
    w_type = np.concatenate([d['tournament'].map(importance_weight).values] * 2)
    w = w_time * w_type
    teams = sorted(set(att)); T = len(teams)
    ai = pd.Categorical(att,  categories=teams).codes
    di = pd.Categorical(deff, categories=teams).codes
    n = len(att)
    rows = np.repeat(np.arange(n), 2)
    cols = np.empty(2 * n); vals = np.empty(2 * n)
    cols[0::2] = ai; vals[0::2] =  1.0
    cols[1::2] = di; vals[1::2] = -1.0
    X_team = sp.csr_matrix((vals, (rows, cols)), shape=(n, T))
    X = sp.hstack([sp.csr_matrix(home.reshape(-1, 1)), X_team]).tocsr()
    m = PoissonRegressor(alpha=ridge, fit_intercept=True, max_iter=3000)
    m.fit(X, goals, sample_weight=w)
    beta0 = float(m.intercept_)
    home_eff = float(m.coef_[0])
    strengths = {t: float(m.coef_[1 + i]) for i, t in enumerate(teams)}
    return beta0, home_eff, strengths

class Model:
    def __init__(self, beta0, home, strengths):
        self.b0, self.h, self.r = beta0, home, strengths
    def lambdas(self, a, b):
        ha = self.h if a in HOSTS else 0.0
        hb = self.h if b in HOSTS else 0.0
        la = np.exp(self.b0 + (self.r[a] - self.r[b]) + ha)
        lb = np.exp(self.b0 + (self.r[b] - self.r[a]) + hb)
        return la, lb

def predict_match(model, a, b, max_goals=10, top_n=10):
    la, lb = model.lambdas(a, b)
    pa = poisson.pmf(np.arange(max_goals + 1), la)
    pb = poisson.pmf(np.arange(max_goals + 1), lb)
    M  = np.outer(pa, pb)

    p_a    = np.tril(M, -1).sum()
    p_draw = np.trace(M)
    p_b    = np.triu(M,  1).sum()

    flat    = M.ravel()
    top_idx = np.argsort(flat)[::-1][:top_n]
    scorelines = []
    for idx in top_idx:
        i, j = np.unravel_index(idx, M.shape)
        scorelines.append({'score': f'{i}-{j}', 'probability': round(100 * flat[idx], 2)})

    # ── goal lines ──────────────────────────────────────────────────────────
    n_g        = max_goals + 1
    idx_arr    = np.arange(n_g)
    total_mask = idx_arr[:, None] + idx_arr[None, :]   # (i+j) at each cell

    def team_overs(lam):
        return {f'over_{k-1}.5': float(1 - poisson.cdf(k - 1, lam)) for k in [1, 2, 3, 4]}

    def total_overs():
        return {f'total_over_{k-1}.5': float(M[total_mask >= k].sum()) for k in [1, 2, 3, 4, 5]}

    # ── margin lines ─────────────────────────────────────────────────────────
    # np.tril(M, -k).sum() = P(A wins by >= k goals)
    # np.triu(M,  k).sum() = P(B wins by >= k goals)
    def margin_overs():
        return {
            **{f'a_margin_over_{k-1}.5': float(np.tril(M, -k).sum()) for k in [1, 2, 3]},
            **{f'b_margin_over_{k-1}.5': float(np.triu(M,  k).sum()) for k in [1, 2, 3]},
        }

    return {
        'team_a': a, 'team_b': b,
        'xg_a': la, 'xg_b': lb,
        'p_a_win': p_a, 'p_draw': p_draw, 'p_b_win': p_b,
        'scorelines': scorelines,
        **{f'a_{k}': v for k, v in team_overs(la).items()},
        **{f'b_{k}': v for k, v in team_overs(lb).items()},
        **total_overs(),
        **margin_overs(),
    }

def print_match_prediction(model, a, b, top_n=10):
    p  = predict_match(model, a, b, top_n=top_n)
    W  = max(52, len(a) + len(b) + 10)
    sep = '─' * W

    print(f'\n{sep}')
    print(f'  {a}  vs  {b}')
    print(sep)
    print(f'  Expected goals :  {p["xg_a"]:.2f}  –  {p["xg_b"]:.2f}')
    print(f'  {a} win  :  {100*p["p_a_win"]:5.1f}%')
    print(f'  Draw           :  {100*p["p_draw"]:5.1f}%')
    print(f'  {b} win  :  {100*p["p_b_win"]:5.1f}%')

    print(f'\n  Top {top_n} most likely scorelines:')
    for rank, s in enumerate(p['scorelines'], 1):
        bar = '█' * int(s['probability'] / 0.5)
        print(f'  {rank:2d}. {s["score"]:>5s}   {s["probability"]:5.2f}%  {bar}')

    # ── goal lines ──────────────────────────────────────────────────────────
    col = max(len(a), len(b), 5) + 2
    print(f'\n  Goal lines (% chance of scoring at least N goals):')
    print(f'  {"":>{col}}   >0.5    >1.5    >2.5    >3.5')
    print(f'  {a:>{col}}   {100*p["a_over_0.5"]:5.1f}%  {100*p["a_over_1.5"]:5.1f}%  {100*p["a_over_2.5"]:5.1f}%  {100*p["a_over_3.5"]:5.1f}%')
    print(f'  {b:>{col}}   {100*p["b_over_0.5"]:5.1f}%  {100*p["b_over_1.5"]:5.1f}%  {100*p["b_over_2.5"]:5.1f}%  {100*p["b_over_3.5"]:5.1f}%')
    print(f'  {"Total":>{col}}   {100*p["total_over_0.5"]:5.1f}%  {100*p["total_over_1.5"]:5.1f}%  {100*p["total_over_2.5"]:5.1f}%  {100*p["total_over_3.5"]:5.1f}%  (>4.5: {100*p["total_over_4.5"]:.1f}%)')

    # ── margin / handicap lines ──────────────────────────────────────────────
    print(f'\n  Winning margin (% chance of winning by at least N goals):')
    print(f'  {"":>{col}}   -0.5    -1.5    -2.5')
    print(f'  {a:>{col}}   {100*p["a_margin_over_0.5"]:5.1f}%  {100*p["a_margin_over_1.5"]:5.1f}%  {100*p["a_margin_over_2.5"]:5.1f}%')
    print(f'  {b:>{col}}   {100*p["b_margin_over_0.5"]:5.1f}%  {100*p["b_margin_over_1.5"]:5.1f}%  {100*p["b_margin_over_2.5"]:5.1f}%')
    print(f'\n{sep}\n')

ALIAS = {'Czechia': 'Czech Republic', 'USA': 'United States',
         'Turkiye': 'Turkey', 'Curacao': 'Curaçao'}
def canon(t): return ALIAS.get(t, t)

GROUPS = {
 'A': ['Mexico','South Korea','South Africa','Czechia'],
 'B': ['Canada','Switzerland','Qatar','Bosnia and Herzegovina'],
 'C': ['Brazil','Morocco','Scotland','Haiti'],
 'D': ['USA','Australia','Paraguay','Turkiye'],
 'E': ['Germany','Ecuador','Ivory Coast','Curacao'],
 'F': ['Netherlands','Japan','Sweden','Tunisia'],
 'G': ['Belgium','Iran','Egypt','New Zealand'],
 'H': ['Spain','Uruguay','Saudi Arabia','Cape Verde'],
 'I': ['France','Senegal','Iraq','Norway'],
 'J': ['Argentina','Algeria','Austria','Jordan'],
 'K': ['Portugal','Colombia','Uzbekistan','DR Congo'],
 'L': ['England','Croatia','Ghana','Panama'],
}
GROUPS = {g: [canon(t) for t in ts] for g, ts in GROUPS.items()}

def predict_group_matches(model):
    rows = []
    for g, teams in GROUPS.items():
        for x in range(len(teams)):
            for y in range(x + 1, len(teams)):
                p = predict_match(model, teams[x], teams[y])
                top = p['scorelines'][0]
                rows.append([g, p['team_a'], p['team_b'],
                             round(p['xg_a'], 2), round(p['xg_b'], 2),
                             round(100 * p['p_a_win'], 1), round(100 * p['p_draw'], 1),
                             round(100 * p['p_b_win'], 1), top['score'], top['probability']])
    return pd.DataFrame(rows, columns=[
        'Group', 'TeamA', 'TeamB', 'xG_A', 'xG_B',
        'A win%', 'Draw%', 'B win%', 'Likely score', 'Score%'])

def predict_group_lines(model):
    """Over/under lines for every group fixture — individual goals, totals, and margins."""
    rows = []
    for g, teams in GROUPS.items():
        for x in range(len(teams)):
            for y in range(x + 1, len(teams)):
                p = predict_match(model, teams[x], teams[y])
                a, b = p['team_a'], p['team_b']
                rows.append({
                    'Group': g, 'TeamA': a, 'TeamB': b,
                    'xG_A': round(p['xg_a'], 2), 'xG_B': round(p['xg_b'], 2),
                    # individual team goal overs
                    'A >0.5g%': round(100 * p['a_over_0.5'], 1),
                    'A >1.5g%': round(100 * p['a_over_1.5'], 1),
                    'A >2.5g%': round(100 * p['a_over_2.5'], 1),
                    'A >3.5g%': round(100 * p['a_over_3.5'], 1),
                    'B >0.5g%': round(100 * p['b_over_0.5'], 1),
                    'B >1.5g%': round(100 * p['b_over_1.5'], 1),
                    'B >2.5g%': round(100 * p['b_over_2.5'], 1),
                    'B >3.5g%': round(100 * p['b_over_3.5'], 1),
                    # total goals overs
                    'Tot >0.5%': round(100 * p['total_over_0.5'], 1),
                    'Tot >1.5%': round(100 * p['total_over_1.5'], 1),
                    'Tot >2.5%': round(100 * p['total_over_2.5'], 1),
                    'Tot >3.5%': round(100 * p['total_over_3.5'], 1),
                    'Tot >4.5%': round(100 * p['total_over_4.5'], 1),
                    # winning margin / handicap
                    'A -0.5%': round(100 * p['a_margin_over_0.5'], 1),
                    'A -1.5%': round(100 * p['a_margin_over_1.5'], 1),
                    'A -2.5%': round(100 * p['a_margin_over_2.5'], 1),
                    'B -0.5%': round(100 * p['b_margin_over_0.5'], 1),
                    'B -1.5%': round(100 * p['b_margin_over_1.5'], 1),
                    'B -2.5%': round(100 * p['b_margin_over_2.5'], 1),
                })
    return pd.DataFrame(rows)

THIRD_SLOTS = {
    74: set('ABCDF'), 77: set('CDFGH'), 79: set('CEFHI'), 80: set('EHIJK'),
    81: set('BEFIJ'), 82: set('AEHIJ'), 85: set('EFGIJ'), 87: set('DEIJL'),
}
R32 = {
 73:('R_A','R_B'), 74:('W_E','T74'), 75:('W_F','R_C'), 76:('W_C','R_F'),
 77:('W_I','T77'), 78:('R_E','R_I'), 79:('W_A','T79'), 80:('W_L','T80'),
 81:('W_D','T81'), 82:('W_G','T82'), 83:('R_K','R_L'), 84:('W_H','R_J'),
 85:('W_B','T85'), 86:('W_J','R_H'), 87:('W_K','T87'), 88:('R_D','R_G'),
}
R16 = {89:(74,77),90:(73,75),91:(76,78),92:(79,80),93:(83,84),94:(81,82),95:(86,88),96:(85,87)}
QF  = {97:(89,90),98:(91,92),99:(93,94),100:(95,96)}
SF  = {101:(97,98),102:(99,100)}
FINAL = (101,102)

def simulate_once(model, group_goals, idx):
    winners, runners = {}, {}
    thirds = []
    for g, teams in GROUPS.items():
        pts = {t:0 for t in teams}; gf={t:0 for t in teams}; ga={t:0 for t in teams}
        for (ga_, gb_, a, b) in group_goals[g]:
            sa, sb = a[idx], b[idx]
            gf[ga_]+=sa; ga[ga_]+=sb; gf[gb_]+=sb; ga[gb_]+=sa
            if sa>sb: pts[ga_]+=3
            elif sb>sa: pts[gb_]+=3
            else: pts[ga_]+=1; pts[gb_]+=1
        order = sorted(teams, key=lambda t:(pts[t], gf[t]-ga[t], gf[t], RNG.random()), reverse=True)
        winners[g], runners[g] = order[0], order[1]
        t3 = order[2]
        thirds.append((g, pts[t3], gf[t3]-ga[t3], gf[t3], t3))
    thirds.sort(key=lambda x:(x[1],x[2],x[3],RNG.random()), reverse=True)
    qual = thirds[:8]
    qual_groups = [q[0] for q in qual]
    third_team = {q[0]: q[4] for q in qual}
    slot_ids = list(THIRD_SLOTS.keys())
    cost = np.ones((8,8))
    for i,s in enumerate(slot_ids):
        for j,grp in enumerate(qual_groups):
            if grp in THIRD_SLOTS[s]: cost[i,j]=0.0
    ri,cj = linear_sum_assignment(cost)
    slot_team = {slot_ids[i]: third_team[qual_groups[j]] for i,j in zip(ri,cj)}
    def resolve(tok):
        if tok.startswith('W_'): return winners[tok[2]]
        if tok.startswith('R_'): return runners[tok[2]]
        if tok.startswith('T'):  return slot_team[int(tok[1:])]
        return tok
    reached = {}
    def mark(team, stage):
        if reached.get(team,0) < stage: reached[team]=stage
    for g in GROUPS:
        mark(winners[g],1); mark(runners[g],1)
    for s in slot_team.values(): mark(s,1)
    def play_ko(a, b, stage_of_winner):
        la, lb = model.lambdas(a,b)
        sa, sb = RNG.poisson(la), RNG.poisson(lb)
        if sa==sb:
            sa2, sb2 = RNG.poisson(la*0.33), RNG.poisson(lb*0.33)
            w = a if sa2>sb2 else (b if sb2>sa2 else (a if RNG.random()<0.5 else b))
        else:
            w = a if sa>sb else b
        mark(w, stage_of_winner)
        return w
    res = {}
    for mid,(L,Rr) in R32.items(): res[mid]=play_ko(resolve(L),resolve(Rr),2)
    for mid,(x,y) in R16.items(): res[mid]=play_ko(res[x],res[y],3)
    for mid,(x,y) in QF.items():  res[mid]=play_ko(res[x],res[y],4)
    for mid,(x,y) in SF.items():  res[mid]=play_ko(res[x],res[y],5)
    champ = play_ko(res[FINAL[0]],res[FINAL[1]],6)
    return reached

def run(model, n_sims=20000):
    group_goals = {}
    for g, teams in GROUPS.items():
        fixtures=[]
        for i in range(len(teams)):
            for j in range(i+1,len(teams)):
                a,b = teams[i],teams[j]
                la,lb = model.lambdas(a,b)
                fixtures.append((a,b, RNG.poisson(la,n_sims), RNG.poisson(lb,n_sims)))
        group_goals[g]=fixtures
    stages=['R32','R16','QF','SF','Final','Champion']
    counts={t:np.zeros(6) for g in GROUPS for t in GROUPS[g]}
    for k in range(n_sims):
        reached = simulate_once(model, group_goals, k)
        for t,s in reached.items():
            for si in range(s): counts[t][si]+=1
    rows=[]
    for t,c in counts.items():
        rows.append([t]+list(c/n_sims*100))
    out=pd.DataFrame(rows, columns=['Team']+stages).sort_values('Champion',ascending=False)
    return out.reset_index(drop=True)

print('Model code loaded.')

In [ ]:
# Fit the model
df = load_matches()
ref_date = pd.Timestamp('2026-06-19')
print(f'Fitting model on {WINDOW_YEARS}y window up to {ref_date.date()} ...')
beta0, home, strengths = fit_strengths(df, ref_date)
model = Model(beta0, home, strengths)
print(f'Done.  beta0 = {beta0:+.3f}   home effect = {home:+.3f}')

In [ ]:
# ── Predict a specific game ───────────────────────────────────────────────────
# Edit the two team names below, then run this cell.
# Use the exact names from the GROUPS dict above (or any team in the dataset).

TEAM_A = 'Spain'
TEAM_B = 'France'
TOP_N  = 10        # how many scorelines to show

print_match_prediction(model, TEAM_A, TEAM_B, top_n=TOP_N)

In [ ]:
# Team strength rankings (all 48 finalists)
finalists = [t for g in GROUPS for t in GROUPS[g]]
rank = sorted(finalists, key=lambda t: strengths.get(t, -9), reverse=True)
strength_df = pd.DataFrame([
    {'Rank': i+1, 'Team': t, 'Strength (r_t)': round(strengths.get(t, float('nan')), 3)}
    for i, t in enumerate(rank)
])
display(strength_df)

In [ ]:
# Group-stage match predictions (all 72 fixtures)
matches = predict_group_matches(model)
display(matches)

# Goal lines + winning-margin lines for every fixture
lines = predict_group_lines(model)
display(lines)

In [ ]:
# Tournament simulation (20,000 runs) — takes ~1 min
print('Simulating 20,000 tournaments ...')
forecast = run(model, n_sims=20000)
display(forecast.style.format({c: '{:.1f}' for c in forecast.columns if c != 'Team'}))

In [ ]:
matches.to_csv('wc2026_match_predictions.csv', index=False)
lines.to_csv('wc2026_lines.csv', index=False)
forecast.to_csv('wc2026_forecast.csv', index=False)
print('CSVs saved to Colab runtime.')

---

## WC 2026: Performance & Live Calibration

**Performance cell** — fits a clean pre-tournament model (June 10) and scores every completed WC 2026 match using RPS (Ranked Probability Score). Lower is better; the naive 1/3-each baseline ≈ 0.222.

**Calibration cell** — refits the model through today, incorporating all completed WC 2026 matches (importance weight 4.0, near-zero time decay), then shows which teams' strength estimates moved the most.

Re-run **cell-2** first to get the latest `results.csv`.

In [ ]:
# ── WC 2026: Pre-tournament Poisson model performance ────────────────────────
# Fits a clean pre-tournament model (ref_date 2026-06-10) — no WC 2026 results
# in training — and evaluates it against every completed WC 2026 match.
# Re-run cell-2 first to download the latest results.csv.
#
# Metrics:
#   RPS     — Ranked Probability Score for 3-way outcome (lower = better, 0 = perfect)
#             ~0.22 = naive 1/3 baseline  |  ~0.19-0.20 = good model  |  ~0.18 = bookmaker level
#   xG MAE  — Mean absolute error of expected goals vs actual goals (per team per match)
#             ~1.0-1.2 = typical for football models (goals are very noisy)
#   Tot MAE — MAE of total goals per match
#             ~1.5-2.0 = typical  |  <1.5 = good
#   Mrg MAE — MAE of goal margin (winning margin prediction)
#             ~1.3-1.7 = typical  |  <1.3 = good
#   Score%  — % where top-1 most likely scoreline was correct
#             ~5-10% is normal (exact scores are hard); >10% is good

WC26_START = pd.Timestamp('2026-06-11')

_MATCH_ALIAS = {
    'Korea Republic': 'South Korea', 'Korea DPR': 'North Korea',
    "Côte d'Ivoire": 'Ivory Coast', "Cote d'Ivoire": 'Ivory Coast',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Bosnia & Herzegovina': 'Bosnia and Herzegovina',
    'IR Iran': 'Iran', 'Turkiye': 'Turkey', 'Czechia': 'Czech Republic',
    'Curacao': 'Curaçao', 'USA': 'United States',
}
def _norm(t): return canon(_MATCH_ALIAS.get(str(t), str(t)))

print('Fitting pre-tournament model (2026-06-10) ...')
beta0_pre, home_pre, str_pre = fit_strengths(df, pd.Timestamp('2026-06-10'))
model_pre = Model(beta0_pre, home_pre, str_pre)
print(f'Done.  beta0 = {beta0_pre:+.3f}   home = {home_pre:+.3f}')

wc26_results = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'] >= WC26_START)
].copy()

if wc26_results.empty:
    print('\nNo WC 2026 results found — re-run cell-2 to download latest data.')
else:
    def _rps(pa, pd_, pb, oa, od, ob):
        return 0.5 * ((pa - oa)**2 + (pa + pd_ - oa - od)**2)

    rows, skipped = [], []
    for _, r in wc26_results.iterrows():
        a   = _norm(r['home_team'])
        b   = _norm(r['away_team'])
        hs  = int(r['home_score'])
        as_ = int(r['away_score'])
        if a not in str_pre or b not in str_pre:
            skipped.append(f'{a} vs {b}')
            continue
        pred = predict_match(model_pre, a, b)
        pa, pd_, pb = pred['p_a_win'], pred['p_draw'], pred['p_b_win']
        oa, od, ob  = int(hs > as_), int(hs == as_), int(hs < as_)
        actual    = 'H' if oa else ('D' if od else 'A')
        top       = max([('H', pa), ('D', pd_), ('A', pb)], key=lambda x: x[1])[0]
        top_score = pred['scorelines'][0]['score']
        rows.append({
            'Date':      r['date'].strftime('%b %d'),
            'Home':      a, 'Score': f'{hs}–{as_}', 'Away': b,
            'Actual':    actual, 'Pred': top,
            'Hit':       '✓' if top == actual else '✗',
            'p(H)':      round(100 * pa,  1),
            'p(D)':      round(100 * pd_, 1),
            'p(A)':      round(100 * pb,  1),
            'RPS':       round(_rps(pa, pd_, pb, oa, od, ob), 4),
            '_naive':    round(_rps(1/3, 1/3, 1/3, oa, od, ob), 4),
            'xG_H':      round(pred['xg_a'], 2),
            'xG_A':      round(pred['xg_b'], 2),
            'xG_tot':    round(pred['xg_a'] + pred['xg_b'], 2),
            'xG_mrg':    round(pred['xg_a'] - pred['xg_b'], 2),
            'act_tot':   hs + as_,
            'act_mrg':   hs - as_,
            'top1':      top_score,
            'score_hit': '✓' if top_score == f'{hs}-{as_}' else '✗',
        })

    if skipped:
        print(f'Skipped (not in model): {", ".join(skipped)}')

    perf_df   = pd.DataFrame(rows)
    avg_rps   = perf_df['RPS'].mean()
    avg_naive = perf_df['_naive'].mean()
    n_hit     = (perf_df['Hit'] == '✓').sum()
    n_score   = (perf_df['score_hit'] == '✓').sum()
    n         = len(perf_df)
    skill     = (avg_naive - avg_rps) / avg_naive * 100

    goals_err = pd.concat([
        (perf_df['xG_H'] - perf_df.apply(lambda r: int(r['Score'].split('–')[0]), axis=1)).abs(),
        (perf_df['xG_A'] - perf_df.apply(lambda r: int(r['Score'].split('–')[1]), axis=1)).abs(),
    ])
    xg_mae_val = goals_err.mean()
    xg_bias    = (perf_df['xG_H'] - perf_df.apply(lambda r: int(r['Score'].split('–')[0]), axis=1)).mean()
    tot_mae    = (perf_df['xG_tot'] - perf_df['act_tot']).abs().mean()
    mrg_mae    = (perf_df['xG_mrg'] - perf_df['act_mrg']).abs().mean()

    def _grade(val, good, great, lower_is_better=True):
        if lower_is_better:
            return '🟢 great' if val <= great else ('🟡 good' if val <= good else '🔴 weak')
        else:
            return '🟢 great' if val >= great else ('🟡 good' if val >= good else '🔴 weak')

    print(f'\n=== Poisson pre-tournament model: {n} WC 2026 matches (out-of-sample ✓) ===')
    print(f'  Outcome RPS      : {avg_rps:.4f}  [{_grade(avg_rps, 0.20, 0.19)}]  '
          f'(naive={avg_naive:.4f}, skill={skill:+.1f}%,  bookmaker≈0.188)')
    print(f'  Correct outcome  : {n_hit}/{n}  ({100*n_hit/n:.0f}%)  [{_grade(100*n_hit/n, 50, 55, lower_is_better=False)}]  '
          f'(naive=33%)')
    print(f'  Correct scoreline: {n_score}/{n}  ({100*n_score/n:.0f}%)  [{_grade(100*n_score/n, 8, 12, lower_is_better=False)}]  '
          f'(typical 5–10%)')
    print(f'  xG MAE (per team): {xg_mae_val:.3f}  [{_grade(xg_mae_val, 1.2, 1.0)}]  '
          f'(typical 1.0–1.2,  bias={xg_bias:+.3f})')
    print(f'  Total goals MAE  : {tot_mae:.3f}  [{_grade(tot_mae, 2.0, 1.5)}]  (typical 1.5–2.0)')
    print(f'  Margin MAE       : {mrg_mae:.3f}  [{_grade(mrg_mae, 1.7, 1.3)}]  (typical 1.3–1.7)')

    display(
        perf_df[['Date','Home','Score','Away','Hit','p(H)','p(D)','p(A)','RPS',
                 'xG_H','xG_A','top1','score_hit']]
        .rename(columns={
            'Hit':       'Outcome?',
            'p(H)':      'P(home)%',
            'p(D)':      'P(draw)%',
            'p(A)':      'P(away)%',
            'xG_H':      'xG (home)',
            'xG_A':      'xG (away)',
            'top1':      'Pred score',
            'score_hit': 'Exact?',
        })
        .style
        .background_gradient(subset=['RPS'], cmap='RdYlGn_r', vmin=0.0, vmax=0.25)
        .format({'P(home)%': '{:.1f}', 'P(draw)%': '{:.1f}', 'P(away)%': '{:.1f}',
                 'RPS': '{:.4f}', 'xG (home)': '{:.2f}', 'xG (away)': '{:.2f}'})
    )

In [ ]:
# ── WC 2026: Live-calibrated Poisson model ───────────────────────────────────
# Refits the Poisson model through today so all completed WC 2026 matches are
# baked into the strength estimates (importance 4.0, near-zero time decay).
# Re-run cell-2 first to pull the latest results.csv.
# Requires the performance cell above to have run (defines str_pre, wc26_results).

import datetime
ref_live = pd.Timestamp(datetime.date.today())
print(f'Fitting live model through {ref_live.date()} ...')
beta0_live, home_live, str_live = fit_strengths(df, ref_live)
model_live = Model(beta0_live, home_live, str_live)
print(f'Done.  beta0 = {beta0_live:+.3f}   home = {home_live:+.3f}')

# Show how team strengths shifted after seeing WC 2026 results
finalists = [_norm(t) for g in GROUPS for t in GROUPS[g]]
diffs = []
for t in finalists:
    r_pre  = str_pre.get(t, 0.0)
    r_live = str_live.get(t, 0.0)
    diffs.append({'Team': t, 'Pre-WC': round(r_pre, 3),
                  'Live': round(r_live, 3), 'Δ': round(r_live - r_pre, 3)})

diff_df = (pd.DataFrame(diffs)
           .sort_values('Δ', ascending=False)
           .reset_index(drop=True))

n_wc26 = len(wc26_results)
print(f'\n=== Strength shifts after {n_wc26} WC 2026 matches ===')
print('Positive Δ = team outperformed pre-tournament expectation; negative = underperformed.')
display(
    diff_df.style
    .background_gradient(subset=['Δ'], cmap='RdYlGn', vmin=-0.3, vmax=0.3)
    .format({'Pre-WC': '{:+.3f}', 'Live': '{:+.3f}', 'Δ': '{:+.3f}'})
)

In [ ]:
# ── Match prediction: live-calibrated model ───────────────────────────────────
# Requires the two cells above to have been run first.
# model_live is refitted through today with all completed WC 2026 matches included.

TEAM_A = 'Spain'
TEAM_B = 'France'
TOP_N  = 10

print(f'Live model (through {ref_live.date()})')
print_match_prediction(model_live, _norm(TEAM_A), _norm(TEAM_B), top_n=TOP_N)

---

## Random Forest Extension (Groll et al. 2019)

Augments the Poisson ranking model with external covariates (squad value, age, caps, UCL players, GDP, population, FIFA rank, bookmaker odds, coach tenure, confederation) trained on 2006–2022 WC match data.

Upload `wc_historical_data.py` and `covariates_2026.py`, then run the three cells below.

In [ ]:
# ── RF 1/3: Upload data files and build training dataset ─────────────────────
import sys
from google.colab import files as colab_files

print('Upload wc_historical_data.py and covariates_2026.py when prompted ...')
colab_files.upload()

for mod in ['wc_historical_data', 'covariates_2026']:
    if mod in sys.modules:
        del sys.modules[mod]
import wc_historical_data as hd
import covariates_2026    as cov

FEAT_COLS = [
    'ability_diff',
    'log_sv',     'log_sv_opp',
    'sq_age',     'sq_age_opp',
    'sq_caps',    'sq_caps_opp',
    'sq_cl',      'sq_cl_opp',
    'log_gdp',    'log_gdp_opp',
    'log_pop',    'log_pop_opp',
    'fifa_rank',  'fifa_rank_opp',
    'odds_prob',  'odds_prob_opp',
    'coach_tenure', 'coach_tenure_opp',
    'is_host', 'same_conf',
]

RESULTS_ALIAS = {
    'Korea Republic':       'South Korea',
    'Korea DPR':            'North Korea',
    "Côte d'Ivoire":        'Ivory Coast',
    "Cote d'Ivoire":        'Ivory Coast',
    'Bosnia-Herzegovina':   'Bosnia and Herzegovina',
    'Bosnia & Herzegovina': 'Bosnia and Herzegovina',
    'IR Iran':              'Iran',
    'Turkiye':              'Turkey',
    'Czechia':              'Czech Republic',
    'Curacao':              'Curaçao',
    'USA':                  'United States',
}
def rname(t): return RESULTS_ALIAS.get(t, t)

def normalize_strengths(raw):
    out = dict(raw)
    for t, v in raw.items():
        out[rname(t)] = v
        out[canon(t)] = v
    return out

def _lg(x): return float(np.log(x)) if (x is not None and x > 0) else np.nan

def build_row(team, opp, year, strengths, norm_probs):
    conf_t = hd.WC_CONFEDERATION.get(team)
    conf_o = hd.WC_CONFEDERATION.get(opp)
    r_t, r_o = strengths.get(team, 0.0), strengths.get(opp, 0.0)
    return {
        'ability_diff':     r_t - r_o,
        'log_sv':           _lg(hd.WC_SQUAD_VALUE.get(year, {}).get(team)),
        'log_sv_opp':       _lg(hd.WC_SQUAD_VALUE.get(year, {}).get(opp)),
        'sq_age':           hd.get_squad_age(team, year),
        'sq_age_opp':       hd.get_squad_age(opp, year),
        'sq_caps':          hd.get_squad_caps(team, year),
        'sq_caps_opp':      hd.get_squad_caps(opp, year),
        'sq_cl':            hd.get_squad_cl(team, year),
        'sq_cl_opp':        hd.get_squad_cl(opp, year),
        'log_gdp':          _lg(hd.get_gdp(team, year)),
        'log_gdp_opp':      _lg(hd.get_gdp(opp, year)),
        'log_pop':          _lg(hd.get_pop(team, year)),
        'log_pop_opp':      _lg(hd.get_pop(opp, year)),
        'fifa_rank':        hd.get_fifa_rank(team, year),
        'fifa_rank_opp':    hd.get_fifa_rank(opp, year),
        'odds_prob':        norm_probs.get(team, np.nan),
        'odds_prob_opp':    norm_probs.get(opp, np.nan),
        'coach_tenure':     hd.get_coach_tenure(team, year),
        'coach_tenure_opp': hd.get_coach_tenure(opp, year),
        'is_host':          int(hd.is_host(team, year)),
        'same_conf':        int(conf_t == conf_o) if (conf_t and conf_o) else 0,
    }

WC_TRAIN = {
    2006: ('2006-06-09', '2006-07-09', '2006-06-08'),
    2010: ('2010-06-11', '2010-07-11', '2010-06-10'),
    2014: ('2014-06-12', '2014-07-13', '2014-06-11'),
    2018: ('2018-06-14', '2018-07-15', '2018-06-13'),
    2022: ('2022-11-20', '2022-12-18', '2022-11-19'),
}

df_matches = load_matches()
all_rows = []

for year, (start_s, end_s, ref_s) in WC_TRAIN.items():
    start, end, ref = map(pd.Timestamp, (start_s, end_s, ref_s))
    _, _, str_y_raw = fit_strengths(df_matches, ref)
    str_y = normalize_strengths(str_y_raw)
    raw = {t: hd.odds_to_prob(o) for t, o in hd.WC_ODDS.get(year, {}).items()}
    tot = sum(raw.values())
    norm = {t: p / tot for t, p in raw.items()} if tot > 0 else {}
    wc = df_matches[(df_matches['tournament'] == 'FIFA World Cup') &
                    (df_matches['date'] >= start) & (df_matches['date'] <= end)]
    for _, row in wc.iterrows():
        ht, at = rname(row['home_team']), rname(row['away_team'])
        hs, as_ = int(row['home_score']), int(row['away_score'])
        for team, opp, goals in [(ht, at, hs), (at, ht, as_)]:
            feat = build_row(team, opp, year, str_y, norm)
            feat['goals'] = goals
            feat['year']  = year
            all_rows.append(feat)

train_df = pd.DataFrame(all_rows)
print(f'Training rows: {len(train_df)}  ({len(WC_TRAIN)} WC years × all matches, 2 rows per fixture)')
print(f'\nMissing-value rate per feature:')
print(train_df[FEAT_COLS].isna().mean().round(3).to_string())

In [ ]:
# ── RF 2/3: Append WC 2026 results to training data ──────────────────────────
# Re-run cell-2 first to download the latest results.csv.
# Run this BEFORE the training cell so the RF includes actual tournament results.
# If the live calibration cell above has been run, str_live is used for
# ability_diff automatically; otherwise falls back to the cell-4 pre-tournament fit.

wc26_start = pd.Timestamp('2026-06-11')

_raw26 = {t: cov.odds_to_prob_2026(t) for t in cov.WC_2026_ODDS}
_tot26 = sum(v for v in _raw26.values() if v is not None)
_norm26 = {t: (p / _tot26 if p is not None else np.nan) for t, p in _raw26.items()}

try:
    _str26 = normalize_strengths(str_live)
    print('Using live-calibrated strengths for ability_diff.')
except NameError:
    _str26 = normalize_strengths(strengths)
    print('Using pre-tournament strengths (run the calibration cell first to use live strengths).')

wc26_played = df_matches[
    (df_matches['tournament'] == 'FIFA World Cup') &
    (df_matches['date'] >= wc26_start)
]

rows_26 = []
for _, row in wc26_played.iterrows():
    ht = canon(rname(row['home_team']))
    at = canon(rname(row['away_team']))
    hs, as_ = int(row['home_score']), int(row['away_score'])
    for team, opp, goals in [(ht, at, hs), (at, ht, as_)]:
        c  = cov.TEAM_COVARIATES.get(team, {})
        co = cov.TEAM_COVARIATES.get(opp,  {})
        conf_t = hd.WC_CONFEDERATION.get(team)
        conf_o = hd.WC_CONFEDERATION.get(opp)
        feat = {
            'ability_diff':     _str26.get(team, 0.0) - _str26.get(opp, 0.0),
            'log_sv':           _lg(c.get('squad_value')),
            'log_sv_opp':       _lg(co.get('squad_value')),
            'sq_age':           c.get('squad_age'),
            'sq_age_opp':       co.get('squad_age'),
            'sq_caps':          c.get('squad_caps'),
            'sq_caps_opp':      co.get('squad_caps'),
            'sq_cl':            c.get('squad_cl'),
            'sq_cl_opp':        co.get('squad_cl'),
            'log_gdp':          _lg(c.get('gdp')),
            'log_gdp_opp':      _lg(co.get('gdp')),
            'log_pop':          _lg(c.get('pop')),
            'log_pop_opp':      _lg(co.get('pop')),
            'fifa_rank':        c.get('fifa_rank'),
            'fifa_rank_opp':    co.get('fifa_rank'),
            'odds_prob':        _norm26.get(team, np.nan),
            'odds_prob_opp':    _norm26.get(opp,  np.nan),
            'coach_tenure':     hd.get_coach_tenure(team, 2026),
            'coach_tenure_opp': hd.get_coach_tenure(opp,  2026),
            'is_host':          int(team in HOSTS),
            'same_conf':        int(conf_t == conf_o) if (conf_t and conf_o) else 0,
            'goals':            goals,
            'year':             2026,
        }
        rows_26.append(feat)

if rows_26:
    n_matches = len(rows_26) // 2
    train_df = pd.concat([train_df, pd.DataFrame(rows_26)], ignore_index=True)
    print(f'Added {n_matches} WC 2026 matches ({len(rows_26)} rows) — total training rows: {len(train_df)}')
else:
    print('No WC 2026 matches found. Re-run cell-2 to download the latest results.csv.')

In [ ]:
# ── Squad value adjustments (injuries / late call-ups) ───────────────────────
# Subtract a player's market value (EUR millions) from their team's squad total.
# Leave the dict empty ({}) to use the original pre-tournament values.
#
# Example:
#   'France':  -180   # Mbappé ruled out (€180M)
#   'England': -120   # Bellingham out  (€120M)

SQUAD_VALUE_ADJUSTMENTS = {
    # 'Team name': change in EUR millions (negative = player out, positive = call-up)
}

# Apply adjustments to a working copy of cov2026 used by RFModel
cov2026_adjusted = {}
for team, vals in cov.TEAM_COVARIATES.items():
    cov2026_adjusted[team] = dict(vals)
    cov2026_adjusted[team]['coach_tenure'] = hd.get_coach_tenure(team, 2026)
    if team in SQUAD_VALUE_ADJUSTMENTS:
        original = cov2026_adjusted[team]['squad_value']
        adjusted = original + SQUAD_VALUE_ADJUSTMENTS[team]
        cov2026_adjusted[team]['squad_value'] = max(adjusted, 1.0)  # floor at 1 to avoid log(0)
        print(f'  {team}: squad_value {original:.1f} → {cov2026_adjusted[team]["squad_value"]:.1f} ({SQUAD_VALUE_ADJUSTMENTS[team]:+.1f})')

if not SQUAD_VALUE_ADJUSTMENTS:
    print('No adjustments applied — using original squad values.')
else:
    print('Done. Re-run the RF train cell and everything below it to apply these adjustments.')

In [ ]:
# ── RF train: fit Poisson Random Forest ──────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute   import SimpleImputer
from sklearn.pipeline import Pipeline

X_train = train_df[FEAT_COLS].values
y_train = train_df['goals'].values.astype(float)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestRegressor(
        n_estimators=1000,
        criterion='poisson',   # minimises Poisson deviance; requires sklearn >= 1.0
        min_samples_leaf=5,    # regularise — training set is only ~500 rows
        max_features='sqrt',
        random_state=20260614,
        n_jobs=-1,
    )),
])

# WC 2026 rows are upweighted — most relevant signal for remaining matches.
# Set WC26_WEIGHT = 1.0 to treat all years equally.
WC26_WEIGHT    = 2.0
sample_weights = np.where(train_df['year'] == 2026, WC26_WEIGHT, 1.0)

pipe.fit(X_train, y_train, rf__sample_weight=sample_weights)
print(f'RF trained.  (WC 2026 rows weighted ×{WC26_WEIGHT})')

imp_df = pd.DataFrame({
    'feature':    FEAT_COLS,
    'importance': pipe.named_steps['rf'].feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('\nFeature importances:')
display(imp_df.style
        .bar(subset='importance', color='steelblue')
        .format({'importance': '{:.4f}'}))

In [ ]:
# ── RF 3/3: RFModel class, run simulation, compare with Poisson ───────────────

# Normalise 2026 outright odds to implied win probabilities
raw26 = {t: cov.odds_to_prob_2026(t) for t in cov.WC_2026_ODDS}
tot26 = sum(v for v in raw26.values() if v is not None)
norm26 = {t: (p / tot26 if p is not None else np.nan) for t, p in raw26.items()}

# Poisson strengths for RFModel — use live-calibrated version if available
try:
    strengths_rf = normalize_strengths(str_live)
    print('RFModel using live-calibrated Poisson strengths.')
except NameError:
    strengths_rf = normalize_strengths(strengths)
    print('RFModel using pre-tournament Poisson strengths (run the calibration cell first to use live strengths).')

class RFModel:
    """Drop-in for Model: same lambdas(a, b) interface. Caches predictions."""

    def __init__(self, pipe, strengths, cov2026, norm26, conf_map):
        self.pipe   = pipe
        self.r      = strengths
        self.cov    = cov2026
        self.odds   = norm26
        self.conf   = conf_map
        self._cache = {}

    def _feat_vec(self, team, opp):
        c  = self.cov.get(team, {})
        co = self.cov.get(opp,  {})
        conf_t = self.conf.get(team)
        conf_o = self.conf.get(opp)
        row = {
            'ability_diff':     self.r.get(team, 0.0) - self.r.get(opp, 0.0),
            'log_sv':           _lg(c.get('squad_value')),
            'log_sv_opp':       _lg(co.get('squad_value')),
            'sq_age':           c.get('squad_age'),
            'sq_age_opp':       co.get('squad_age'),
            'sq_caps':          c.get('squad_caps'),
            'sq_caps_opp':      co.get('squad_caps'),
            'sq_cl':            c.get('squad_cl'),
            'sq_cl_opp':        co.get('squad_cl'),
            'log_gdp':          _lg(c.get('gdp')),
            'log_gdp_opp':      _lg(co.get('gdp')),
            'log_pop':          _lg(c.get('pop')),
            'log_pop_opp':      _lg(co.get('pop')),
            'fifa_rank':        c.get('fifa_rank'),
            'fifa_rank_opp':    co.get('fifa_rank'),
            'odds_prob':        self.odds.get(team, np.nan),
            'odds_prob_opp':    self.odds.get(opp,  np.nan),
            'coach_tenure':     c.get('coach_tenure'),
            'coach_tenure_opp': co.get('coach_tenure'),
            'is_host':          int(team in HOSTS),
            'same_conf':        int(conf_t == conf_o) if (conf_t and conf_o) else 0,
        }
        return [row[f] if row[f] is not None else np.nan for f in FEAT_COLS]

    def lambdas(self, a, b):
        if (a, b) not in self._cache:
            X = np.array([self._feat_vec(a, b), self._feat_vec(b, a)], dtype=float)
            preds = self.pipe.predict(X)
            la, lb = max(float(preds[0]), 0.01), max(float(preds[1]), 0.01)
            self._cache[(a, b)] = (la, lb)
            self._cache[(b, a)] = (lb, la)
        return self._cache[(a, b)]


rf_model = RFModel(pipe, strengths_rf, cov2026_adjusted, norm26, hd.WC_CONFEDERATION)

print('\nRF vs Poisson expected goals:')
print(f'  {"Match":<40}  {"RF":^11}  {"Poisson":^11}')
for a, b in [('Spain','France'), ('Argentina','Brazil'),
             ('United States','Mexico'), ('England','Germany')]:
    la, lb   = rf_model.lambdas(a, b)
    lpa, lpb = model.lambdas(a, b)
    print(f'  {a} vs {b:<20}  RF {la:.2f}–{lb:.2f}  Poi {lpa:.2f}–{lpb:.2f}')

print('\nSimulating 20,000 tournaments with RF model (≈2–3 min) ...')
rf_forecast = run(rf_model, n_sims=20000)

stages = ['R32', 'R16', 'QF', 'SF', 'Final', 'Champion']
merged = (
    forecast[['Team'] + stages]
    .merge(
        rf_forecast[['Team'] + stages].rename(columns={s: f'RF_{s}' for s in stages}),
        on='Team', how='outer',
    )
    .sort_values('Champion', ascending=False)
    .reset_index(drop=True)
)
fmt = {c: '{:.1f}' for c in merged.columns if c != 'Team'}

print('\n── Poisson vs RF model: Champion % ──')
display(merged[['Team', 'Champion', 'RF_Champion', 'Final', 'RF_Final']]
        .style.format(fmt)
        .background_gradient(cmap='YlGn', subset=['Champion', 'RF_Champion']))

print('\n── Full RF forecast ──')
display(rf_forecast.style.format(fmt))

In [ ]:
# ── RF model performance on WC 2026 ──────────────────────────────────────────
# Requires: RF 1/3 → RF 3/3 cells to have been run.
# Also requires the Poisson performance cell (cell-11) to have run, since it
# defines wc26_results, _norm, and _rps.
#
# ⚠️  CONTAMINATION WARNING
# If you ran the "Append WC 2026 results" cell (RF 2/3) before training, the
# RF has seen some of these games during training → metrics will be IN-SAMPLE
# and artificially good. The cell detects and flags this automatically.
#
# For a clean out-of-sample evaluation: skip the "Append WC 2026" cell,
# retrain (RF train), then run this cell.
#
# Metrics (same benchmarks as Poisson performance cell):
#   RPS     — ~0.22 = naive  |  ~0.19-0.20 = good  |  ~0.188 = bookmaker level
#   Correct outcome  — naive=33%  |  ≥50% = good  |  ≥55% = great
#   Correct scoreline — ~5-10% typical  |  ≥10% = good  |  ≥12% = great
#   xG MAE (per team) — ~1.0-1.2 typical  |  ≤1.2 = good  |  ≤1.0 = great
#   Total goals MAE   — ~1.5-2.0 typical  |  ≤2.0 = good  |  ≤1.5 = great
#   Margin MAE        — ~1.3-1.7 typical  |  ≤1.7 = good  |  ≤1.3 = great

def _grade(val, good, great, lower_is_better=True):
    if lower_is_better:
        return '🟢 great' if val <= great else ('🟡 good' if val <= good else '🔴 weak')
    else:
        return '🟢 great' if val >= great else ('🟡 good' if val >= good else '🔴 weak')

try:
    rf_model
    _rf_ready = True
except NameError:
    _rf_ready = False
    print('rf_model not found — run RF 1/3 → RF 3/3 first, then re-run this cell.')

if _rf_ready:
    # Detect contamination
    wc26_in_train = 'year' in train_df.columns and (train_df['year'] == 2026).any()
    n_wc26_train  = int((train_df['year'] == 2026).sum() // 2) if wc26_in_train else 0

    if wc26_in_train:
        print(f'⚠️  IN-SAMPLE: {n_wc26_train} WC 2026 matches were in RF training data.')
        print('   Metrics below are optimistic. Skip "Append WC 2026" cell and retrain for clean eval.\n')
    else:
        print('✓  OUT-OF-SAMPLE: RF trained on 2006–2022 only.\n')

    rows_rf, skipped_rf = [], []
    for _, r in wc26_results.iterrows():
        a   = _norm(r['home_team'])
        b   = _norm(r['away_team'])
        hs  = int(r['home_score'])
        as_ = int(r['away_score'])
        try:
            la, lb = rf_model.lambdas(a, b)
        except Exception:
            skipped_rf.append(f'{a} vs {b}')
            continue
        pred = predict_match(rf_model, a, b)
        pa, pd_, pb = pred['p_a_win'], pred['p_draw'], pred['p_b_win']
        oa, od, ob  = int(hs > as_), int(hs == as_), int(hs < as_)
        actual    = 'H' if oa else ('D' if od else 'A')
        top       = max([('H', pa), ('D', pd_), ('A', pb)], key=lambda x: x[1])[0]
        top_score = pred['scorelines'][0]['score']
        rows_rf.append({
            'Date':      r['date'].strftime('%b %d'),
            'Home':      a, 'Score': f'{hs}–{as_}', 'Away': b,
            'Actual':    actual, 'Pred': top,
            'Hit':       '✓' if top == actual else '✗',
            'p(H)':      round(100 * pa,  1),
            'p(D)':      round(100 * pd_, 1),
            'p(A)':      round(100 * pb,  1),
            'RPS':       round(_rps(pa, pd_, pb, oa, od, ob), 4),
            '_naive':    round(_rps(1/3, 1/3, 1/3, oa, od, ob), 4),
            'xG_H':      round(la, 2),
            'xG_A':      round(lb, 2),
            'xG_tot':    round(la + lb, 2),
            'xG_mrg':    round(la - lb, 2),
            'act_tot':   hs + as_,
            'act_mrg':   hs - as_,
            'top1':      top_score,
            'score_hit': '✓' if top_score == f'{hs}-{as_}' else '✗',
        })

    if skipped_rf:
        print(f'Skipped (not in model): {", ".join(skipped_rf)}')

    rf_perf   = pd.DataFrame(rows_rf)
    avg_rps   = rf_perf['RPS'].mean()
    avg_naive = rf_perf['_naive'].mean()
    n_hit     = (rf_perf['Hit'] == '✓').sum()
    n_score   = (rf_perf['score_hit'] == '✓').sum()
    n         = len(rf_perf)
    skill     = (avg_naive - avg_rps) / avg_naive * 100

    goals_err_rf = pd.concat([
        (rf_perf['xG_H'] - rf_perf.apply(lambda r: int(r['Score'].split('–')[0]), axis=1)).abs(),
        (rf_perf['xG_A'] - rf_perf.apply(lambda r: int(r['Score'].split('–')[1]), axis=1)).abs(),
    ])
    xg_mae_rf  = goals_err_rf.mean()
    xg_bias_rf = (rf_perf['xG_H'] - rf_perf.apply(lambda r: int(r['Score'].split('–')[0]), axis=1)).mean()
    tot_mae_rf = (rf_perf['xG_tot'] - rf_perf['act_tot']).abs().mean()
    mrg_mae_rf = (rf_perf['xG_mrg'] - rf_perf['act_mrg']).abs().mean()

    label = 'IN-SAMPLE ⚠️' if wc26_in_train else 'out-of-sample ✓'
    print(f'=== RF model: {n} WC 2026 matches ({label}) ===')
    print(f'  Outcome RPS      : {avg_rps:.4f}  [{_grade(avg_rps, 0.20, 0.19)}]  '
          f'(naive={avg_naive:.4f}, skill={skill:+.1f}%,  bookmaker≈0.188)')
    print(f'  Correct outcome  : {n_hit}/{n}  ({100*n_hit/n:.0f}%)  [{_grade(100*n_hit/n, 50, 55, lower_is_better=False)}]  '
          f'(naive=33%)')
    print(f'  Correct scoreline: {n_score}/{n}  ({100*n_score/n:.0f}%)  [{_grade(100*n_score/n, 8, 12, lower_is_better=False)}]  '
          f'(typical 5–10%)')
    print(f'  xG MAE (per team): {xg_mae_rf:.3f}  [{_grade(xg_mae_rf, 1.2, 1.0)}]  '
          f'(typical 1.0–1.2,  bias={xg_bias_rf:+.3f})')
    print(f'  Total goals MAE  : {tot_mae_rf:.3f}  [{_grade(tot_mae_rf, 2.0, 1.5)}]  (typical 1.5–2.0)')
    print(f'  Margin MAE       : {mrg_mae_rf:.3f}  [{_grade(mrg_mae_rf, 1.7, 1.3)}]  (typical 1.3–1.7)')

    # Side-by-side vs Poisson
    if 'perf_df' in dir():
        poi_rps = perf_df['RPS'].mean()
        print(f'\n── vs Poisson pre-tournament model ──')
        print(f'  RPS:   RF {avg_rps:.4f}  vs  Poisson {poi_rps:.4f}  '
              f'({"RF better" if avg_rps < poi_rps else "Poisson better"})')

    display(
        rf_perf[['Date','Home','Score','Away','Hit','p(H)','p(D)','p(A)','RPS',
                 'xG_H','xG_A','top1','score_hit']]
        .rename(columns={
            'Hit':       'Outcome?',
            'p(H)':      'P(home)%',
            'p(D)':      'P(draw)%',
            'p(A)':      'P(away)%',
            'xG_H':      'xG (home)',
            'xG_A':      'xG (away)',
            'top1':      'Pred score',
            'score_hit': 'Exact?',
        })
        .style
        .background_gradient(subset=['RPS'], cmap='RdYlGn_r', vmin=0.0, vmax=0.25)
        .format({'P(home)%': '{:.1f}', 'P(draw)%': '{:.1f}', 'P(away)%': '{:.1f}',
                 'RPS': '{:.4f}', 'xG (home)': '{:.2f}', 'xG (away)': '{:.2f}'})
    )

In [ ]:
# ── RF feature vector inspector ───────────────────────────────────────────────
# Shows feature importances alongside the actual values for a given matchup.
# Team names are auto-normalised (e.g. Czechia → Czech Republic, USA → United States).

TEAM_A = 'Spain'
TEAM_B = 'France'

TEAM_A = canon(TEAM_A)
TEAM_B = canon(TEAM_B)

la_rf,  lb_rf  = rf_model.lambdas(TEAM_A, TEAM_B)
la_poi, lb_poi = model.lambdas(TEAM_A, TEAM_B)

print(f'Expected goals — RF: {la_rf:.2f}–{lb_rf:.2f}   Poisson: {la_poi:.2f}–{lb_poi:.2f}')
print()

importances = pipe.named_steps['rf'].feature_importances_

rows = []
for feat, imp, val_a, val_b in zip(
    FEAT_COLS,
    importances,
    rf_model._feat_vec(TEAM_A, TEAM_B),
    rf_model._feat_vec(TEAM_B, TEAM_A),
):
    rows.append({
        'feature':    feat,
        'importance': imp,
        TEAM_A:       val_a,
        TEAM_B:       val_b,
    })

feat_df = (pd.DataFrame(rows)
             .sort_values('importance', ascending=False)
             .reset_index(drop=True))

display(feat_df.style
        .format({'importance': '{:.4f}', TEAM_A: '{:.4f}', TEAM_B: '{:.4f}'})
        .bar(subset='importance', color='steelblue')
        .set_caption(f'{TEAM_A} vs {TEAM_B} — RF features sorted by importance'))

In [ ]:
# ── Match prediction ─────────────────────────────────────────────────────────
# Edit the team names and MODEL below, then run this cell.

TEAM_A = 'Spain'
TEAM_B = 'France'
TOP_N  = 10

# MODEL options:
#   'poisson_pre'  — pre-tournament Poisson (requires cell-4)
#   'poisson_live' — live-calibrated Poisson (requires cell-12)
#   'rf'           — Random Forest (requires RF 1/3 → RF 3/3)
MODEL = 'poisson_live'

_a = canon(TEAM_A)
_b = canon(TEAM_B)

if MODEL == 'poisson_pre':
    _m = model
    print(f'Poisson — pre-tournament')
elif MODEL == 'poisson_live':
    _m = model_live
    print(f'Poisson — live-calibrated (through {ref_live.date()})')
elif MODEL == 'rf':
    _m = rf_model
    print('RF model')
else:
    raise ValueError(f"Unknown MODEL {MODEL!r}. Choose 'poisson_pre', 'poisson_live', or 'rf'.")

print_match_prediction(_m, _a, _b, top_n=TOP_N)